# 02 - Fetch GTFS data (lines, direction, calendar, stop times)

Fetches everything the pipeline needs from SEPTA's historical GTFS releases, one download per release (333 @ final run Aug 2026), then pushes 5 new files to /data:

- `2_gtfs_linkages_since_2017_clean.parquet` -- train_number/line/service_id/trip_id/direction_id per release.
- `2_gtfs_active_service_dates.parquet` -- which service_id(s) were active on each calendar date, expanded from each release's `calendar.txt` + `calendar_dates.txt`
- `2_gtfs_calendar_raw.parquet` -- raw `(date, service_id, exception_type)` rows across all releases
- `2_gtfs_stop_times.parquet` -- raw per-release stop_times (`trip_id`, `stop_id`, `stop_sequence`, `arrival_time`, `departure_time`), filtered to rail trip_ids only. Filters `google_rail.zip` which isn't always rail-only weirdly. ~28% of raw `stop_times.txt` rows across all releases were bus data.
- `2_gtfs_stops.parquet` -- raw per-release `stops.txt` (`stop_id`, `stop_name`, `stop_lat`, `stop_lon`), for cross-release stop-id crosswalk 

Pulls everything in one pass + avoids repeated bandwidth costs.

## Installs and imports

In [1]:
import os
import time  # for rate limiting

import pandas as pd

from utils import fetch_gtfs_releases, fetch_gtfs_release_zip

BASEPATH = "../data"

## GTFS: historical API fetch

Produces the four checkpoints above for use later in the pipeline (`02b_gtfs_merge.ipynb` most immediately).

In [ ]:
# helper function
# for a single GTFS release, extracts everything the pipeline 
# needs in one download:
#   - crosswalk: block_id/line/service_id/trip_id/direction_id
#     
#   - active_service_dates: which service_id(s) were active on 
#     each calendar date this release's calendar.txt covers.
#     Uses each service_id's own validity window because the releases'
#     published dates can precede actual calendar fx by days/weeks.

#   - calendar_raw: raw (date, service_id, exception_type) rows from
#     calendar_dates.txt, for 05_calendar_events.ipynb's system-wide
#     service-exception flags 

#   - stop_times: raw (trip_id, stop_id, stop_sequence, arrival_time,
#     departure_time) rows, filtered to rail trip_ids only

#   - stops: raw (stop_id, stop_name, stop_lat, stop_lon) rows -- not
#     filtered to rail, since stop_id alone doesn't identify mode here;
#     the rail-only filtering already applied to trip_ids/stop_times
#     naturally excludes non-rail stops downstream via the crosswalk join

def get_release_gtfs_data(tag, date):
    rail_zip = fetch_gtfs_release_zip(tag)
    if rail_zip is None:
        return None, None, None, None, None

    # some older releases' CSVs have different spacing which breaks read
    read_kwargs = dict(dtype = str, skipinitialspace = True)

    trips = pd.read_csv(rail_zip.open("trips.txt"), **read_kwargs,
                        usecols = ["route_id", "service_id", "trip_id",
                                   "direction_id", "block_id"])
    routes = pd.read_csv(rail_zip.open("routes.txt"), **read_kwargs,
                         usecols = ["route_id", "route_long_name", "route_type"])
    routes = routes[routes["route_type"] == "2"]  # commuter rail only

    crosswalk = (trips.merge(routes, on = "route_id")[
        ["block_id", "route_long_name", "service_id", "trip_id", "direction_id"]]
                 .drop_duplicates())
    crosswalk["gtfs_date"] = date

    calendar = pd.read_csv(rail_zip.open("calendar.txt"), **read_kwargs)
    calendar.columns = calendar.columns.str.strip()
    calendar["start_date"] = pd.to_datetime(calendar["start_date"], format = "%Y%m%d")
    calendar["end_date"] = pd.to_datetime(calendar["end_date"], format = "%Y%m%d")

    cal_dates = pd.read_csv(rail_zip.open("calendar_dates.txt"), **read_kwargs)
    cal_dates.columns = cal_dates.columns.str.strip()
    cal_dates["date"] = pd.to_datetime(cal_dates["date"], format = "%Y%m%d")
    cal_dates["exception_type"] = cal_dates["exception_type"].astype(int)

    calendar_raw = cal_dates[["date", "service_id", "exception_type"]].copy()
    calendar_raw["gtfs_release_date"] = date

    # expand each service_id's weekday flags across its own date range
    day_cols = ["monday", "tuesday", "wednesday", "thursday",
                "friday", "saturday", "sunday"]
    rows = []
    for _, svc in calendar.iterrows():
        all_dates = pd.date_range(svc["start_date"], svc["end_date"])
        for day_name in day_cols:
            if svc[day_name] != "1":
                continue
            matching = all_dates[all_dates.day_name().str.lower() == day_name]
            rows.extend((d, svc["service_id"]) for d in matching)
    base = pd.DataFrame(rows, columns = ["service_date", "service_id"])
    base["service_date"] = pd.to_datetime(base["service_date"])

    # apply calendar_dates.txt exceptions: add (1) or remove (2) specific dates
    added = cal_dates[cal_dates["exception_type"] == 1].rename(
        columns = {"date": "service_date"})[["service_date", "service_id"]]
    removed = cal_dates[cal_dates["exception_type"] == 2].rename(
        columns = {"date": "service_date"})[["service_date", "service_id"]]

    active = pd.concat([base, added], ignore_index = True).drop_duplicates()
    active = active.merge(removed, on = ["service_date", "service_id"],
                          how = "left", indicator = True)
    active = active[active["_merge"] == "left_only"].drop(columns = "_merge")
    active["source_gtfs_date"] = date

    rail_trip_ids = set(crosswalk["trip_id"])
    stop_times = pd.read_csv(rail_zip.open("stop_times.txt"), **read_kwargs,
                             usecols = ["trip_id", "stop_id", "stop_sequence",
                                        "arrival_time", "departure_time"])
    stop_times = stop_times[stop_times["trip_id"].isin(rail_trip_ids)]
    stop_times["gtfs_date"] = date

    stops = pd.read_csv(rail_zip.open("stops.txt"), **read_kwargs,
                        usecols = ["stop_id", "stop_name", "stop_lat", "stop_lon"])
    stops["stop_lat"] = stops["stop_lat"].astype(float)
    stops["stop_lon"] = stops["stop_lon"].astype(float)
    stops["gtfs_date"] = date

    return crosswalk, active, calendar_raw, stop_times, stops

### Full fetch (first time, or full refresh)

In [ ]:
# NOTE: first GTFS release in 2017 is Jan 21 -- pull from last 2016 release
# to capture early Jan 2017 train assignments
all_releases = fetch_gtfs_releases()

print(f"Latest release: {all_releases[0]['date']}")
print(f"Oldest release: {all_releases[-1]['date']}")
print(f"Number of releases: {len(all_releases)}")

Latest release: 2026-07-23
Oldest release: 2016-12-16
Number of releases: 333


Checkpoints progress every `CHECKPOINT_EVERY` releases to `data/.gtfs_fetch_partial_*.parquet`, and resumes from there if re-run.

In [4]:
CHECKPOINT_EVERY = 25

checkpoint_paths = {
    "crosswalk": f"{BASEPATH}/.gtfs_fetch_partial_crosswalk.parquet",
    "active_service_dates": f"{BASEPATH}/.gtfs_fetch_partial_active_service_dates.parquet",
    "calendar_raw": f"{BASEPATH}/.gtfs_fetch_partial_calendar_raw.parquet",
    "stop_times": f"{BASEPATH}/.gtfs_fetch_partial_stop_times.parquet",
    "stops": f"{BASEPATH}/.gtfs_fetch_partial_stops.parquet",
}

if os.path.exists(checkpoint_paths["crosswalk"]):
    raw_crosswalks = [pd.read_parquet(checkpoint_paths["crosswalk"])]
    raw_active = [pd.read_parquet(checkpoint_paths["active_service_dates"])]
    raw_calendar = [pd.read_parquet(checkpoint_paths["calendar_raw"])]
    raw_stop_times = [pd.read_parquet(checkpoint_paths["stop_times"])]
    raw_stops = [pd.read_parquet(checkpoint_paths["stops"])] if os.path.exists(checkpoint_paths["stops"]) else []
    done_dates = set(raw_crosswalks[0]["gtfs_date"].unique())
    print(f"Resuming from checkpoint: {len(done_dates)} releases already fetched")
else:
    raw_crosswalks, raw_active, raw_calendar, raw_stop_times, raw_stops = [], [], [], [], []
    done_dates = set()

# download each release and extract the crosswalk/calendar/stop_times/stops tables
# loops thru all and skips if failed
for i, rel in enumerate(all_releases):
    if rel["date"] in done_dates:
        continue
    try:
        cw, active, cal_raw, st, stops = get_release_gtfs_data(rel["tag"], rel["date"])
        if cw is not None:
            raw_crosswalks.append(cw)
            raw_active.append(active)
            raw_calendar.append(cal_raw)
            raw_stop_times.append(st)
            raw_stops.append(stops)
    except Exception as e:
        print(f"Skipped {rel['tag']}: {e}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.concat(raw_crosswalks, ignore_index = True).drop_duplicates().to_parquet(checkpoint_paths["crosswalk"])
        pd.concat(raw_active, ignore_index = True).drop_duplicates().to_parquet(checkpoint_paths["active_service_dates"])
        pd.concat(raw_calendar, ignore_index = True).drop_duplicates().to_parquet(checkpoint_paths["calendar_raw"])
        pd.concat(raw_stop_times, ignore_index = True).drop_duplicates().to_parquet(checkpoint_paths["stop_times"])
        pd.concat(raw_stops, ignore_index = True).drop_duplicates().to_parquet(checkpoint_paths["stops"])
        print(f"[{i+1}/{len(all_releases)}] {rel['date']} -- checkpointed ({len(raw_crosswalks)} releases fetched so far)")

    time.sleep(0.5)

[25/333] 2025-08-25 -- checkpointed (25 releases fetched so far)


[50/333] 2024-09-06 -- checkpointed (49 releases fetched so far)


[75/333] 2024-02-07 -- checkpointed (74 releases fetched so far)


[100/333] 2023-05-27 -- checkpointed (99 releases fetched so far)


[125/333] 2022-09-02 -- checkpointed (124 releases fetched so far)


...

Skipped v202005171: "There is no item named 'calendar_dates.txt' in the archive"


[250/333] 2020-04-06 -- checkpointed (244 releases fetched so far)


[275/333] 2019-08-28 -- checkpointed (269 releases fetched so far)


[300/333] 2018-10-11 -- checkpointed (294 releases fetched so far)


[325/333] 2017-06-08 -- checkpointed (319 releases fetched so far)


## Clean, combine, and save

In [5]:
# clean fetched crosswalk data
# combine raw downloads, de-duplicate, rename columns
crosswalk = (
    pd.concat(raw_crosswalks, ignore_index = True) # combine
    .drop_duplicates()                       # drop dups
    .astype({"block_id": str})               # block id -> str
    .rename(columns = {"block_id": "train_number", # rename cols
                       "route_long_name": "line"})
)

crosswalk["gtfs_date"] = crosswalk["gtfs_date"].astype("datetime64[ns]")

crosswalk.to_parquet(f"{BASEPATH}/2_gtfs_linkages_since_2017_clean.parquet")
print(f"Saved {len(crosswalk):,} crosswalk rows")

Saved 366,522 crosswalk rows


In [ ]:
# clean active calendar data 
active_service_dates = (
    pd.concat(raw_active, ignore_index = True)
    .drop_duplicates()
)

active_service_dates["service_date"] = active_service_dates["service_date"].astype("datetime64[ns]")
active_service_dates["source_gtfs_date"] = active_service_dates["source_gtfs_date"].astype("datetime64[ns]")

active_service_dates.to_parquet(f"{BASEPATH}/2_gtfs_active_service_dates.parquet")
print(f"Saved {len(active_service_dates):,} active service-date rows")

Saved 34,009 active service-date rows


In [ ]:
# clean raw calendar dates data (for 05_calendar_events.ipynb)
calendar_raw = (
    pd.concat(raw_calendar, ignore_index = True)
    .drop_duplicates()
)

calendar_raw["date"] = calendar_raw["date"].astype("datetime64[ns]")
calendar_raw["gtfs_release_date"] = calendar_raw["gtfs_release_date"].astype("datetime64[ns]")

calendar_raw.to_parquet(f"{BASEPATH}/2_gtfs_calendar_raw.parquet")
print(f"Saved {len(calendar_raw):,} raw calendar_dates rows")

Saved 1,315 raw calendar_dates rows


In [ ]:
# clean fetched stop times  data 
stop_times = (
    pd.concat(raw_stop_times, ignore_index = True)
    .drop_duplicates()
)

stop_times["gtfs_date"] = stop_times["gtfs_date"].astype("datetime64[ns]")

stop_times.to_parquet(f"{BASEPATH}/2_gtfs_stop_times.parquet")
print(f"Saved {len(stop_times):,} stop_times rows")

Saved 5,392,855 stop_times rows


In [ ]:
# clean fetched stops data (for 02c_gtfs_stops_crosswalk.ipynb)
stops_all = (
    pd.concat(raw_stops, ignore_index = True)
    .drop_duplicates()
)

stops_all["gtfs_date"] = stops_all["gtfs_date"].astype("datetime64[ns]")

stops_all.to_parquet(f"{BASEPATH}/2_gtfs_stops.parquet")
print(f"Saved {len(stops_all):,} stops rows")

Saved 55,295 stops rows


In [10]:
# clean up partial checkpoint files once full run succeeded
for path in checkpoint_paths.values():
    if os.path.exists(path):
        os.remove(path)